# 03 · แผนที่จุดเสี่ยงเชิงสัมพัทธ์ (หลักฐานชุด "ที่ไหน")

1. ค่าจากรอบวัดเคลื่อนที่ (12 รอบ เริ่มและจบที่สถานี) ÷ ค่าสถานีในชั่วโมงเดียวกัน = **ratio**
2. รวมเป็นกริด 100 ม. ใช้ค่ามัธยฐานของ ratio ต่อช่อง
3. ส่ง `grid_cells.csv` ให้ `analysis/src/spatial_compare.py` เทียบ IDW / Kriging / Random Forest ด้วย spatial block CV

ลองระบบด้วย `data/sample/demo_grid_cells.csv` (ข้อมูลจำลอง) ก่อนได้

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# หา root ของ repo (โฟลเดอร์ที่มี data/) ไม่ว่าจะเปิด notebook จากที่ไหน
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

In [ ]:
from pyproj import Transformer
GRID_M = 100
dev = pd.read_csv(PROC / "device_readings.csv", parse_dates=["time"], index_col="time")
st = pd.read_csv(PROC / "hourly_khonkaen.csv", parse_dates=["time"], index_col="time")["station_pm25"]

mob = dev.dropna(subset=["lat", "lng", "pm25_corr"]).copy()
mob["station"] = st.reindex(mob.index.floor("h")).to_numpy()
mob = mob[mob["station"] > 0]
mob["ratio"] = mob["pm25_corr"] / mob["station"]

tr = Transformer.from_crs("EPSG:4326", "EPSG:32648", always_xy=True)
x, y = tr.transform(mob["lng"].to_numpy(), mob["lat"].to_numpy())
mob["gx"], mob["gy"] = (x // GRID_M).astype(int), (y // GRID_M).astype(int)
cells = mob.groupby(["gx", "gy"]).agg(ratio=("ratio", "median"), n=("ratio", "size"),
                                      runs=("ratio", lambda s: pd.Index(s.index.date).nunique())).reset_index()
inv = Transformer.from_crs("EPSG:32648", "EPSG:4326", always_xy=True)
cells["lon"], cells["lat"] = inv.transform((cells["gx"] + 0.5) * GRID_M, (cells["gy"] + 0.5) * GRID_M)
cells = cells[cells["n"] >= 3]                         # ตัดช่องที่มีข้อมูลน้อยเกินไป
cells[["lat", "lon", "ratio", "n"]].to_csv(PROC / "grid_cells.csv", index=False)
print(len(cells), "ช่องกริด")

In [ ]:
sc = plt.scatter(cells["lon"], cells["lat"], c=cells["ratio"], cmap="RdYlGn_r", vmin=0.6, vmax=1.6, s=18)
plt.colorbar(sc, label="ratio เทียบสถานี (1 = เท่าสถานี)")
plt.gca().set_aspect("equal"); plt.title("PM2.5 เชิงสัมพัทธ์ รายช่อง 100 ม."); plt.show()

เทียบวิธีประมาณค่าเชิงพื้นที่:

```
python analysis/src/spatial_compare.py data/processed/grid_cells.csv
```